# Lezione 41 — LoRA su Gemma

> **Modulo:** LoRA e adattamento efficiente · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 35 (inferenza Gemma), Lezione 40 (LoRA da zero).
>
> Obiettivo pratico unico: attivare LoRA su Gemma via KerasHub
> (`backbone.enable_lora(rank=...)`) e capire cosa cambia rispetto alla LoRA
> costruita a mano nella Lezione 40.
>
> ⚠️ **Nota di ambiente**: le celle Gemma sono **guardate** e saltate qui
> (pesi gated). Il codice e' l'API reale; la parte di progetto e' eseguibile.
> Vedi `course/research_gaps.md`.

## Teoria essenziale

Nella Lezione 40 abbiamo inserito a mano $B,A$ dentro un singolo strato. Su un
modello vero come Gemma non serve riscrivere gli strati: KerasHub espone
`backbone.enable_lora(rank=r)`, che **inietta** gli adapter LoRA negli strati di
attenzione, **congela** i pesi originali e rende addestrabili solo gli adapter.
Poi si addestra con il normale `fit`, ma il numero di parametri addestrabili
crolla — esattamente il risparmio calcolato nella Lezione 39.

In [1]:
# Guardia di ambiente (come nelle Lezioni 34-37): KerasHub + pesi Gemma sono un
# extra opzionale e richiedono download autenticato di diversi GB. Qui vengono
# saltati; il notebook resta eseguibile in CI. Con GPU e Gemma configurato, la
# stessa cella gira davvero.
GEMMA_AVAILABLE = False
_motivo = ""
try:
    import keras  # noqa: F401
    import keras_hub  # noqa: F401
    GEMMA_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    _motivo = f"{type(exc).__name__}: {exc}"
print("KerasHub disponibile." if GEMMA_AVAILABLE
      else f"KerasHub/Gemma NON disponibile ({_motivo or 'assente'}); celle modello saltate.")

KerasHub/Gemma NON disponibile (ModuleNotFoundError: No module named 'keras'); celle modello saltate.


In [2]:
if GEMMA_AVAILABLE:
    import keras_hub
    gemma = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")
    # inietta LoRA (rank 4) negli strati di attenzione: congela il resto
    gemma.backbone.enable_lora(rank=4)
    gemma.summary()  # i "Trainable params" ora sono una frazione minima
else:
    print("[saltato] gemma.backbone.enable_lora(rank=4)")
    print("Inietterebbe adapter LoRA rank-4, congelando i pesi originali.")

[saltato] gemma.backbone.enable_lora(rank=4)
Inietterebbe adapter LoRA rank-4, congelando i pesi originali.


## Il progetto: Memory AI Lab, passo 41 — stima del risparmio

Anche senza il modello posso stimare il risparmio su uno strato di attenzione di
dimensione tipica di Gemma-2B, riusando la matematica della Lezione 39. E' la
parte eseguibile e verificabile.

In [3]:
def stima_trainable_lora(d_model, n_strati_attn, r, matrici_per_strato=4):
    # ogni strato di attenzione ha ~4 proiezioni (q,k,v,o) d_model x d_model
    pieni = n_strati_attn * matrici_per_strato * d_model * d_model
    lora = n_strati_attn * matrici_per_strato * r * (d_model + d_model)
    return {"pieni": pieni, "lora": lora,
            "percentuale": round(100 * lora / pieni, 4),
            "fattore": round(pieni / lora, 1)}

# Gemma-2B: d_model ~2048, ~18 blocchi (valori d'ordine di grandezza)
s = stima_trainable_lora(d_model=2048, n_strati_attn=18, r=4)
assert s["lora"] < s["pieni"] / 100     # meno dell'1% addestrabile
print("stima risparmio LoRA su attenzione Gemma-2B (r=4):", s)
print(f"-> addestri circa il {s['percentuale']}% dei parametri di attenzione")

stima risparmio LoRA su attenzione Gemma-2B (r=4): {'pieni': 301989888, 'lora': 1179648, 'percentuale': 0.3906, 'fattore': 256.0}
-> addestri circa il 0.3906% dei parametri di attenzione


## Riepilogo (max 8 punti)

1. Su Gemma non si riscrivono gli strati: `backbone.enable_lora(rank=r)`.
2. Inietta adapter LoRA negli strati di attenzione e congela il resto.
3. Si addestra col normale `fit`, ma i parametri addestrabili crollano.
4. E' la stessa idea della Lezione 40, applicata a un modello reale.
5. Il risparmio segue la matematica della Lezione 39: $r(d+k)$ vs $d\cdot k$.
6. Con $r=4$ su Gemma-2B si addestra ben meno dell'1% dei pesi di attenzione.
7. Gli adapter risultanti sono piccoli e portabili (Lezione 44).
8. La qualita' va confrontata con una baseline (Lezione 43).

## Quiz

1. Cosa fa `enable_lora` oltre a iniettare gli adapter?
2. Perche' non serve riscrivere gli strati come nella Lezione 40?
3. Da cosa dipende il numero di parametri addestrabili?

*(Risposte: 1. congela i pesi originali e rende addestrabili solo gli adapter;
2. KerasHub li inietta automaticamente negli strati di attenzione; 3. dal rango
$r$, dalla dimensione degli strati e da quanti strati sono adattati.)*